### Kansas City, St. Louis, and Springfield

In [33]:
import pandas as pd
import numpy as np
import re

In [34]:
kc = pd.read_csv(
    'raw_data/kcmo.csv',
    dtype={"Zip":"string"})
stl = pd.read_csv('raw_data/stl.csv')
spfd = pd.read_csv('raw_data/springfield.csv')

#### Add unique ID's & Market info to each county csv

In [35]:
#KC

# Create APN column
kc["APN"] = (
    "KCMO."
    + kc["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + kc["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

#Create Market column
kc["Market"] = "Kansas City"

#Create State column
kc["State"] = "Missouri"


In [36]:
#STL

# Create APN column
stl["APN"] = (
    "STL_MO."
    + stl["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + stl["Zip"].astype(str)
)

#Create Market column
stl["Market"] = "St. Louis"

#Create State column
stl["State"] = "Missouri"


In [37]:
#Springfield

# Create APN column
spfd["APN"] = (
    "KCMO."
    + spfd["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + spfd["Zip"].astype(str)
)

#Create Market column
spfd["Market"] = "Kansas City"

#Create State column
spfd["State"] = "Missouri"

In [38]:
spfd.head()

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market,State
0,Greene County,10-13-2026,14:25,3457 W Lombard St,Springfield,65802,McKenzie Tuttle and Addison Cramer,"1947 S Ingram Mill Rd Springfield, MO 65804",3,2,...,203800,NaN,NaN,13-21-306-038,Active,NaN,NaN,KCMO.3457_W_LOMBARD_ST_65802,Kansas City,Missouri
1,Greene County,10-14-2026,11:00,1019 W Lynn Street,Springfield,65802,Keishawn Grant,"1019 W Lynn St, Springfield, MO 65802",2,1,...,176200,190896.0,193903.0,13-14-202-009,Active,NaN,NaN,KCMO.1019_W_LYNN_STREET_65802,Kansas City,Missouri
2,Greene County,10-14-2026,12:00,3634 S Kansas Ave,Springfield,65807,Jimmie Bell,"PO Box 11111 Springfield, MO 65808",3,2,...,228600,NaN,NaN,8.81811E+11,Active,NaN,NaN,KCMO.3634_S_KANSAS_AVE_65807,Kansas City,Missouri
3,Greene County,10-20-2026,14:25,117 Moonlight Valley Dr,Ash Grove,65604,Alisha Fuqua,"117 Moonlight Valley Dr, Ash Grove, MO 65604",4,2,...,263300,242291.0,246565.0,06-16-300-031,Active,NaN,NaN,KCMO.117_MOONLIGHT_VALLEY_DR_65604,Kansas City,Missouri
4,Greene County,10-21-2026,11:00,2932 S Marlan Ave,Springfield,65804,Doris Emerson,"3277 S Farm Road 203 Springfield, MO 65809",3,2,...,320700,54024.0,114000.0,19-05-111-012,Active,NaN,NaN,KCMO.2932_S_MARLAN_AVE_65804,Kansas City,Missouri


#### Combine all data to one csv

In [39]:
kc_stl_import = pd.concat([kc, stl, spfd], ignore_index=True)

kc_stl_import.to_csv("export_data/kc_stl_import.csv", index=False)

In [40]:
#Find duplicate APNs

duplicate_apns = kc_stl_import[
    kc_stl_import.duplicated(subset="APN", keep=False)
]

# View them
duplicate_apns.sort_values("APN")

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market,State


In [41]:
#Remove duplicate apns

kc_stl_import.drop_duplicates(['APN'], keep = 'first', inplace=True)

#### Add Columns
County, Last Name

In [42]:
#Full address
#Labeled as 'Last Name' for Salesforce Lead object purposes

kc_stl_import["Last Name"] = (
    kc_stl_import["Address"].fillna("").astype(str)
    + ", "
    + kc_stl_import["City"].fillna("").astype(str)
    + ", MO "
    + kc_stl_import["Zip"].fillna("").astype(str)
)

kc_stl_import["Last Name"] = kc_stl_import["Last Name"]

In [43]:
#County
#All of the values contain county at the end, so I want to remove that, and add the correct record names

kc_stl_import["County"] = kc_stl_import["County"].str.replace(" County", "", regex=False) 

kc_stl_import["County"] = kc_stl_import["County"] + " - MO"

In [44]:
#Change currency columns to numbers

for col in ["Opening Bid"]:
    kc_stl_import.loc[:, col] = (
        kc_stl_import[col]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

In [45]:
kc_stl_import.loc[
    kc_stl_import["Status"] == "Canceled",
    ["Status", "Opening Bid"]
].head(20)

,Status,Opening Bid
0,Canceled,189113.00
4,Canceled,NaN
21,Canceled,NaN
24,Canceled,65878.65
33,Canceled,NaN
68,Canceled,NaN
69,Canceled,NaN
93,Canceled,NaN
94,Canceled,NaN
95,Canceled,NaN


In [46]:
#Update Auction Status 

kc_stl_import["Status"] = np.where(
    kc_stl_import["Status"] == "Canceled",
    "Cancelled",
    ""
)

In [47]:
#Update Opening Bid

kc_stl_import["Opening Bid"] = kc_stl_import["Opening Bid"].astype("string")

kc_stl_import.loc[
    kc_stl_import["Status"].str.strip().str.upper() == "CANCELLED",
    "Opening Bid"
] = ""

In [48]:
kc_stl_import.loc[
    kc_stl_import["Status"] == "Cancelled",
    ["Status", "Opening Bid"]
].head(20)

,Status,Opening Bid
0,Cancelled,
4,Cancelled,
21,Cancelled,
24,Cancelled,
33,Cancelled,
68,Cancelled,
69,Cancelled,
93,Cancelled,
94,Cancelled,
95,Cancelled,


In [49]:
#Change column names
kc_stl_import = kc_stl_import.rename(columns={
    "Date": "Auction Date Updated",
    "Time": "Auction Sale Time",
    "Owner Name": "Property Owner Name",
    "Square Feet": "Bldg Sq Ft",
    "Trustee": "Auction Trustee",
    "File #": "NS Trustee Sale Number",
    "Opening Bid": "Auction Opening Bid",
    "Status": "Auction Status",
    "Bedrooms": "Beds",
    "Bathrooms": "Baths",
    "APN Number": "Tax ID",
    "Assessed Value": "Estimated Value",
    "Original Loan Amount": "Mortgage $"
})

#### Auction Trustee Lookups

In [50]:
contacts = pd.read_csv('sf_data/trusteecontacts.csv')

In [51]:
# Clean up column names if there are leading/trailing spaces
#contacts.columns = contacts.columns.str.strip()

In [52]:
#Normalize text for Contact Name and Associated Names

def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = s.replace("\u00A0", " ")              # non-breaking spaces
    s = re.sub(r"[^\w\s]", " ", s)            # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()        # collapse spaces
    return s

contacts["contact_name_norm"] = contacts["Contact: Contact Name"].map(norm)
contacts["assoc_norm"] = contacts["Associated Names"].map(norm)  # can be blank

In [53]:
#Normalize trustee names in the auction data

kc_stl_import["trustee_norm"] = kc_stl_import["Auction Trustee"].map(norm)

In [54]:
#Find exact matches

exact_map = dict(zip(contacts["contact_name_norm"], contacts["Contact: Contact Name"]))
kc_stl_import["Auction Trustee (Lookup)"] = kc_stl_import["trustee_norm"].map(exact_map)

In [55]:
#Find contains matches

def contains_in_contact_name(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    # find first contact where contact name contains trustee value
    hits = contacts.loc[
        contacts["contact_name_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = kc_stl_import["Auction Trustee (Lookup)"].isna() & (kc_stl_import["trustee_norm"] != "")
kc_stl_import.loc[mask, "Auction Trustee (Lookup)"] = kc_stl_import.loc[mask, "trustee_norm"].map(contains_in_contact_name)

In [56]:
#Find contains matches in associated names

def contains_in_associated(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    hits = contacts.loc[
        contacts["assoc_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = kc_stl_import["Auction Trustee (Lookup)"].isna() & (kc_stl_import["trustee_norm"] != "")
kc_stl_import.loc[mask, "Auction Trustee (Lookup)"] = kc_stl_import.loc[mask, "trustee_norm"].map(contains_in_associated)

In [57]:
#Collect and show unmatched rows

unmatched = (
    kc_stl_import.loc[kc_stl_import["Auction Trustee (Lookup)"].isna(),["Last Name", "Auction Trustee", "trustee_norm"]])

unmatched

,Last Name,Auction Trustee,trustee_norm
6,"1311 E 24th St, Kansas City, MO 64108","singers corp, By: /s/ Michael P. Gaughan",SINGERS CORP BY S MICHAEL P GAUGHAN
61,"925 Grand Blvd, Kansas City, MO 64106",CENTRE TRUSTEE CORP. By: Michele M. OMalley,CENTRE TRUSTEE CORP BY MICHELE M OMALLEY
80,"13513 S Sam Moore Rd, Lone Jack, MO 64070","HB TRUSTEE, INC By Michael D. Fielding",HB TRUSTEE INC BY MICHAEL D FIELDING
176,"5340 Delmar Blvd, St. Louis, MO 63112",William J. Easley BC,WILLIAM J EASLEY BC
187,"5328 Delmar Blvd, St. Louis, MO 63112","William J. Easley BC TRUSTEE SERVICES, INC",WILLIAM J EASLEY BC TRUSTEE SERVICES INC
239,"3940 Highway 79, O Fallon, MO 63366","Trust. MMR TRUSTEE, LLC By: /s/ Philip D. Spei...",TRUST MMR TRUSTEE LLC BY S PHILIP D SPEICHER
264,"2527 Golden Gate Dr, Imperial, MO 63052","Dayid L Marshak, Sheriff",DAYID L MARSHAK SHERIFF
290,"3634 S Kansas Ave, Springfield, MO 65807","SouthLaw, P.C",SOUTHLAW P C


In [58]:
#Confirm what should match but didnt

missing = (
    kc_stl_import.loc[kc_stl_import["Auction Trustee (Lookup)"].isna(), "trustee_norm"]
    .dropna().drop_duplicates()
)

# which of these are present as a normalized contact name?
present_mask = missing.isin(contacts["contact_name_norm"])
missing_not_in_contacts = missing[~present_mask]

missing_not_in_contacts.head(50)

6               SINGERS CORP BY S MICHAEL P GAUGHAN
61         CENTRE TRUSTEE CORP BY MICHELE M OMALLEY
80             HB TRUSTEE INC BY MICHAEL D FIELDING
176                             WILLIAM J EASLEY BC
187        WILLIAM J EASLEY BC TRUSTEE SERVICES INC
239    TRUST MMR TRUSTEE LLC BY S PHILIP D SPEICHER
264                         DAYID L MARSHAK SHERIFF
290                                    SOUTHLAW P C
Name: trustee_norm, dtype: object

In [59]:
kc_stl_import.to_csv("export_data/kc_stl_import.csv", index=False)